# 1. Instalación de Dependencias y Servidor Ollama
Instalamos las librerías necesarias y levantamos el servidor local de Ollama en segundo plano para poder usar el modelo sin salir de Colab.

In [ ]:
# Instalamos dependencias de Python
!pip install gradio pypdf langchain langchain-huggingface langchain-community langchain-ollama langchain-text-splitters chromadb sentence-transformers -q

# Instalamos Ollama en el entorno de Linux de Colab
!curl -fsSL https://ollama.com/install.sh | sh

# Levantamos el servidor en segundo plano y descargamos el modelo
!nohup ollama serve > ollama.log 2>&1 &
!ollama pull gemma4:e2b

# 2. Importaciones y Configuración Base
Importamos todas las herramientas necesarias y creamos el directorio de trabajo donde se almacenarán nuestros corpus en formato JSON.

In [1]:
import gradio as gr
import os
import json
import re
import shutil
import platform
import subprocess
from pypdf import PdfReader

# LangChain & Ollama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Directorio de datos
CARPETA_DATOS = "data"
os.makedirs(CARPETA_DATOS, exist_ok=True)
print(f"✅ Carpeta base '{CARPETA_DATOS}' verificada/creada.")

C:\Users\admin\AppData\Local\Temp\ipykernel_16228\561001749.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


✅ Carpeta base 'data' verificada/creada.


# 3. Detección de Hardware e Inicialización de Modelos
El sistema detecta automáticamente si hay GPUs disponibles para asignar el contexto adecuado. Luego, carga en memoria el modelo de embeddings multilingüe y conecta con el LLM local.

In [ ]:
print("[INFO] Detectando hardware para configurar el LLM...")
sistema = platform.system()
maquina = platform.machine()

tiene_cuda = False
if sistema in ("Linux", "Windows"):
    try:
        resultado = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        tiene_cuda = resultado.returncode == 0
    except FileNotFoundError:
        tiene_cuda = False

es_apple_silicon = (sistema == "Darwin" and maquina == "arm64")

if es_apple_silicon:
    MODEL_NAME = "gemma4:e2b"
    NUM_CTX = 4096
elif tiene_cuda:
    MODEL_NAME = "granite4:latest"
    NUM_CTX = 8192
else:
    MODEL_NAME = "gemma4:e2b"  # Configuración estándar
    NUM_CTX = 2048

print(f"[OK] Modelo asignado: {MODEL_NAME} (Contexto: {NUM_CTX} tokens)")

print("[INFO] Cargando el modelo de embeddings (HuggingFace)...")
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': False}
)

print("[INFO] Conectando con el servidor local de Ollama...")
llm = OllamaLLM(
    model=MODEL_NAME,
    temperature=0.1,
    num_ctx=NUM_CTX
)
print("Modelos listos para operar.")

[INFO] Detectando hardware para configurar el LLM...
[OK] Modelo asignado: gemma4:e2b (Contexto: 2048 tokens)
[INFO] Cargando el modelo de embeddings (HuggingFace)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[INFO] Conectando con el servidor local de Ollama...
🚀 Modelos listos para operar.


# 4. Gestión de Archivos y Extracción PDF
Funciones encargadas de leer los PDFs, limpiar el texto y guardarlos/eliminarlos como archivos JSON en el disco local.

In [ ]:
def listar_corpus_disponibles():
    archivos = [f for f in os.listdir(CARPETA_DATOS) if f.endswith('.json')]
    return archivos if archivos else ["No hay corpus creados aún"]

def extraer_y_limpiar_pdf(ruta_pdf, nombre_original):
    documentos = []
    try:
        reader = PdfReader(ruta_pdf)
        for i, pagina in enumerate(reader.pages):
            texto = pagina.extract_text()
            if texto:
                texto_limpio = re.sub(r'\s+', ' ', texto).strip()
                if texto_limpio:
                    doc = {
                        "contenido": texto_limpio,
                        "metadata": {"fuente": nombre_original, "pagina": i + 1}
                    }
                    documentos.append(doc)
    except Exception as e:
        print(f"[ERROR] No se pudo leer el PDF {nombre_original}: {e}")
    return documentos

def procesar_pdf(archivos_pdf, nombre_nuevo_corpus, corpus_existente, accion):
    if not archivos_pdf:
        return "Por favor, subí al menos un PDF.", gr.update(), gr.update(), gr.update()
    
    if accion == "Crear nuevo corpus":
        if not nombre_nuevo_corpus:
            return "Ingresá un nombre para el nuevo corpus.", gr.update(), gr.update(), gr.update()
        nombre_archivo = f"{nombre_nuevo_corpus.strip().replace(' ', '_')}.json"
        ruta_json = os.path.join(CARPETA_DATOS, nombre_archivo)
        corpus_data = []
    else:
        if not corpus_existente or corpus_existente == "No hay corpus creados aún":
            return "Seleccioná un corpus válido existente.", gr.update(), gr.update(), gr.update()
        ruta_json = os.path.join(CARPETA_DATOS, corpus_existente)
        try:
            with open(ruta_json, 'r', encoding='utf-8') as f:
                corpus_data = json.load(f)
        except Exception as e:
            return f"Error al leer el corpus existente: {e}", gr.update(), gr.update(), gr.update()

    for pdf in archivos_pdf:
        ruta_temp = pdf.name
        nombre_original = os.path.basename(ruta_temp.replace('\\', '/')) 
        nuevos_documentos = extraer_y_limpiar_pdf(ruta_temp, nombre_original)
        corpus_data.extend(nuevos_documentos)
    
    try:
        with open(ruta_json, 'w', encoding='utf-8') as f:
            json.dump(corpus_data, f, ensure_ascii=False, indent=2)
        
        # Borra el caché vectorial si el corpus se actualiza
        nombre_sin_ext = os.path.basename(ruta_json).replace(".json", "")
        ruta_vector_cache = os.path.join("chroma_db", nombre_sin_ext)
        if os.path.exists(ruta_vector_cache):
            shutil.rmtree(ruta_vector_cache)
            
    except Exception as e:
        return f"Error al guardar el JSON: {e}", gr.update(), gr.update(), gr.update()
    
    lista_actualizada = listar_corpus_disponibles()
    return "Corpus guardado exitosamente.", gr.update(choices=lista_actualizada), gr.update(choices=lista_actualizada), gr.update(choices=lista_actualizada)

def eliminar_corpus(corpus_seleccionado):
    if not corpus_seleccionado or corpus_seleccionado == "No hay corpus creados aún":
        return "Seleccioná un corpus para eliminar.", gr.update(), gr.update(), gr.update()
    
    ruta_json = os.path.join(CARPETA_DATOS, corpus_seleccionado)
    nombre_sin_ext = corpus_seleccionado.replace(".json", "")
    ruta_vector_db = os.path.join("chroma_db", nombre_sin_ext)
    
    try:
        if os.path.exists(ruta_json):
            os.remove(ruta_json)
        if os.path.exists(ruta_vector_db):
            shutil.rmtree(ruta_vector_db)
            
        lista_actualizada = listar_corpus_disponibles()
        return f"Corpus '{corpus_seleccionado}' eliminado.", gr.update(choices=lista_actualizada, value=None), gr.update(choices=lista_actualizada, value=None), gr.update(choices=lista_actualizada, value=None)
    except Exception as e:
        return f"Error al eliminar: {e}", gr.update(), gr.update(), gr.update()

# 5. Core RAG: Indexación y Pipeline LCEL (LangChain Expression Language)
Acá definimos las variables globales que mantienen el estado de la aplicación interactiva y las funciones que vectorizan el texto y le dan formato a las respuestas.

In [ ]:
# Variables globales de estado
vector_store = None
retriever_activo = None
pipeline_rag = None
corpus_activo_nombre = None

def formatear_documentos(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def cargar_corpus_en_rag(corpus_seleccionado):
    global vector_store, retriever_activo, pipeline_rag, corpus_activo_nombre

    if not corpus_seleccionado or corpus_seleccionado == "No hay corpus creados aún":
        return "Seleccioná un corpus válido primero."

    ruta_json = os.path.join(CARPETA_DATOS, corpus_seleccionado)
    nombre_sin_ext = corpus_seleccionado.replace(".json", "")
    ruta_vector_db = os.path.join("chroma_db", nombre_sin_ext)

    # 1. PROMPT MEJORADO (Evita que el LLM se asuste y lo obliga a leer todo)
    TEMPLATE_PROMPT = """Eres un analista de datos experto. Tu única tarea es responder la Pregunta del usuario utilizando ÚNICAMENTE la información provista en los Documentos de contexto.

INSTRUCCIONES CLAVE:
1. Lee TODOS los documentos de contexto detenidamente antes de responder.
2. Si encuentras la respuesta, redáctala de forma clara y directa.
3. Si la respuesta definitiva no está, pero hay información relacionada, resúmela y aclara que es un dato parcial.
4. Solo di "No encontré la información" si realmente no hay nada útil en el texto.

Documentos de contexto:
{context}

Pregunta del usuario: {question}

Respuesta analítica:"""

    prompt = PromptTemplate(template=TEMPLATE_PROMPT, input_variables=["context", "question"])

    def construir_pipeline():
        global pipeline_rag, retriever_activo, corpus_activo_nombre
        
        # 2. IMPLEMENTACIÓN DE MMR (Maximal Marginal Relevance)
        # Trae 20 fragmentos candidatos de la base, pero se queda con los 5 más relevantes y diversos
        retriever_activo = vector_store.as_retriever(
            search_type="mmr",
            search_kwargs={"k": 5, "fetch_k": 20} 
        )

        # Aplicamos el filtro por umbral (0.62) y traemos hasta 10 fragmentos (k=10)
        #retriever_activo = vector_store.as_retriever(
        #    search_type="similarity_score_threshold",
        #    search_kwargs={
        #        "k": 10,
        #        "score_threshold": 0.62
        #    }
        #)
        
        pipeline_rag = (
            {"context": retriever_activo | formatear_documentos, "question": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
        )
        corpus_activo_nombre = nombre_sin_ext

    # Carga desde caché si existe
    if os.path.exists(ruta_vector_db) and len(os.listdir(ruta_vector_db)) > 0:
        try:
            vector_store = Chroma(persist_directory=ruta_vector_db, embedding_function=embeddings_model)
            construir_pipeline()
            return f"¡Corpus '{nombre_sin_ext}' "
        except Exception as e:
            return f"Error al cargar almacenamiento persistente: {e}"

    # Indexación por primera vez
    try:
        with open(ruta_json, "r", encoding="utf-8") as f:
            datos_corpus = json.load(f)

        if not datos_corpus:
            return "El archivo JSON seleccionado está vacío."

        documentos_langchain = [Document(page_content=doc["contenido"], metadata=doc["metadata"]) for doc in datos_corpus]
        
        # 3. CHUNKS MÁS GENEROSOS
        splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=300)
        documentos_fragmentados = splitter.split_documents(documentos_langchain)

        vector_store = Chroma.from_documents(
            documents=documentos_fragmentados,
            embedding=embeddings_model,
            persist_directory=ruta_vector_db,
            collection_metadata={"hnsw:space": "cosine"}
        )
        
        construir_pipeline()
        return f"Indexación completada. {len(documentos_fragmentados)} fragmentos procesados."
    except Exception as e:
        return f"Error crítico al indexar: {e}"
    
def responder_rag_interactivo(mensaje, historial):
    global pipeline_rag, retriever_activo, corpus_activo_nombre
    if pipeline_rag is None or retriever_activo is None:
        return "⚠️ El sistema RAG no está activo. Cargá un corpus primero en la pestaña de Configuración."

    try:
        docs_fuentes = retriever_activo.invoke(mensaje)
        fuentes_usadas = f"\n\n--- 📄 FUENTES UTILIZADAS ({corpus_activo_nombre}) ---"
        if docs_fuentes:
            for i, doc in enumerate(docs_fuentes, 1):
                fuentes_usadas += f"\n• [Ref {i}] Archivo: {doc.metadata.get('fuente')} - Pág. {doc.metadata.get('pagina')}"
        else:
            fuentes_usadas += "\n• No se encontraron fragmentos relevantes."

        respuesta_llm = pipeline_rag.invoke(mensaje)
        return f"{respuesta_llm}{fuentes_usadas}"
    except Exception as e:
        return f"❌ Error en la consulta: {str(e)}"

### Primer test

**Vamos a probar el buscador "en crudo" antes de levantar la interfaz gráfica**
**vamos a hacer una consulta sobre sostenibilidad**

In [29]:
import os
from langchain_community.vectorstores import Chroma

# --- 1. CONFIGURACIÓN DEL TEST ---
# IMPORTANTE: Poné el nombre de tu corpus sin la extensión .json (ej: "leyes", "economia")
CORPUS_A_TESTEAR = "Informe_Sustentabilidad_2025" 
query = "¿Cuáles son los principales desafíos del cambio climático mencionados en los informes?"
#query = "¿en que categorías se distribuyen los países?"
#query = "¿por quien coordinado este documento?"
#query = "¿Qué se menciona sobre las emisiones de gases de efecto invernadero?"

ruta_vector_db = os.path.join("chroma_db", CORPUS_A_TESTEAR)

# Verificamos que la base de datos exista
if not os.path.exists(ruta_vector_db):
    print(f"Error: No se encontró la base de datos en {ruta_vector_db}")
else:
    # --- 2. CONEXIÓN A CHROMA Y CONFIGURACIÓN DEL RETRIEVER ---
    print(f"[INFO] Conectando a la base de datos de '{CORPUS_A_TESTEAR}'...")
    
    # Asume que 'embeddings_model' ya fue inicializado en celdas anteriores
    vector_store_test = Chroma(
        persist_directory=ruta_vector_db, 
        embedding_function=embeddings_model
    )
    
    # Aplicamos la configuración MMR para traer los 5 más relevantes y diversos
    retriever_test = vector_store_test.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 10, "fetch_k": 20}
    )

    # --- 3. EJECUCIÓN DE LA BÚSQUEDA ---
    print("[INFO] Buscando fragmentos...\n")
    fragmentos_recuperados = retriever_test.invoke(query)

    print("="*80)
    print(f"Pregunta realizada: '{query}'")
    print(f"Se recuperaron {len(fragmentos_recuperados)} fragmentos relevantes.")
    print("="*80)

    # --- 4. VISUALIZACIÓN DE RESULTADOS ---
    for i, doc in enumerate(fragmentos_recuperados, 1):
        fuente = doc.metadata.get('fuente', 'Desconocida')
        pagina = doc.metadata.get('pagina', 'N/A')
        
        print(f"\n[FRAGMENTO {i} | Fuente: {fuente} | Pág: {pagina}]")
        
        # Mostramos el contenido completo para analizar si el dato exacto está adentro
        contenido = doc.page_content
        print(contenido)
        print("-" * 80)
        
        # Verificación visual rápida: Buscamos si la palabra "Pacto" está en el fragmento
        if "Pacto" in contenido or "Digital" in contenido or "2024" in contenido:
            print("ALERTA DE COINCIDENCIA: Este fragmento parece contener palabras clave de tu pregunta.")
            print("-" * 80)

[INFO] Conectando a la base de datos de 'Informe_Sustentabilidad_2025'...
[INFO] Buscando fragmentos...

Pregunta realizada: '¿Cuáles son los principales desafíos del cambio climático mencionados en los informes?'
Se recuperaron 10 fragmentos relevantes.

[FRAGMENTO 1 | Fuente: informe CENIA CEPAL 2025.pdf | Pág: 163]
y dispositivos de corta vida útil tienen una huella de carbono significativa. Sin políticas que articulen sustentabilidad, eficiencia energética y economía circular, la expansión de la IA puede entrar en contradicción con las metas climáticas regionales. En la medida que la construcción de centros de datos es fundamental para mejorar la capacidad de cómputo de los países, se hace cada vez más relevante incorporar medidas de mitigación para su impacto climático. El aprovechamiento de fuentes de energía renovables no convencionales puede ser un primer paso para utilizar las capacidades de los países para el despliegue de una IA limpia y ecológica. 09 PROMOCIÓN ADECUADA DE C

### Búsqueda con Filtros Avanzados (Metadata Filtering)

**LangChain permite pasarle condiciones a ChromaDB a través del parámetro search_kwargs.**

In [30]:
import os
from langchain_community.vectorstores import Chroma

# --- 1. CONFIGURACIÓN DEL TEST ---
# IMPORTANTE: Reemplazá con el nombre de tu corpus (sin .json)
CORPUS_A_TESTEAR = "Informe_Sustentabilidad_2025" 

query_filtrada = "Estrategias de mitigación y transición energética"

# El nombre EXACTO del archivo o fuente que quedó guardado en los metadatos
fuente_a_filtrar = "informe CENIA CEPAL 2025.pdf" 

ruta_vector_db = os.path.join("chroma_db", CORPUS_A_TESTEAR)

# Verificamos que la base de datos exista
if not os.path.exists(ruta_vector_db):
    print(f"Error: No se encontró la base de datos en {ruta_vector_db}")
else:
    # --- 2. CONEXIÓN A CHROMA ---
    print(f"[INFO] Conectando a la base de datos de '{CORPUS_A_TESTEAR}'...")
    
    # Conectamos con el modelo de embeddings ya inicializado en tu Colab
    vector_store_test = Chroma(
        persist_directory=ruta_vector_db, 
        embedding_function=embeddings_model
    )
    
    # --- 3. CONFIGURACIÓN DEL RETRIEVER CON FILTRO ---
    configuracion_filtros = {
        "k": 3,
        "filter": {"fuente": fuente_a_filtrar} 
    }

    # Inicializamos el retriever aplicando el filtro en search_kwargs
    retriever_filtrado = vector_store_test.as_retriever(
        search_type="similarity",
        search_kwargs=configuracion_filtros
    )

    # --- 4. EJECUCIÓN DE LA BÚSQUEDA ---
    print(f"[INFO] Buscando fragmentos filtrados por fuente '{fuente_a_filtrar}'...\n")
    fragmentos_filtrados = retriever_filtrado.invoke(query_filtrada)

    print("="*80)
    print(f"Pregunta: '{query_filtrada}'")
    print(f"Se recuperaron {len(fragmentos_filtrados)} fragmentos (Filtro: {fuente_a_filtrar}).")
    print("="*80)

    # --- 5. VISUALIZACIÓN DE RESULTADOS ---
    if len(fragmentos_filtrados) == 0:
         print("No se encontraron fragmentos. Verificá que el nombre de la fuente sea exactamente igual al del metadata.")
    else:
        for i, doc in enumerate(fragmentos_filtrados, 1):
            fuente_real = doc.metadata.get('fuente', 'Desconocida')
            pagina = doc.metadata.get('pagina', 'N/A')
            
            print(f"\n[FRAGMENTO {i}]")
            print(f"Fuente Real: {fuente_real} | Pág: {pagina}")
            # Mostramos los primeros 300 caracteres para inspección
            print(f"Contenido:\n{doc.page_content[:300]}...") 
            print("-" * 80)

[INFO] Conectando a la base de datos de 'Informe_Sustentabilidad_2025'...
[INFO] Buscando fragmentos filtrados por fuente 'informe CENIA CEPAL 2025.pdf'...

Pregunta: 'Estrategias de mitigación y transición energética'
Se recuperaron 3 fragmentos (Filtro: informe CENIA CEPAL 2025.pdf).

[FRAGMENTO 1]
Fuente Real: informe CENIA CEPAL 2025.pdf | Pág: 174
Contenido:
ILIA 2025 174 y un tercero que mide la proporción de energías renovables no convencionales (ERNC) dentro de la matriz energética. Con estas modificaciones, el ILIA 2025 busca entregar información comparativa respecto a temas que son de creciente interés dentro de la dimensión de Gobernanza a nivel m...
--------------------------------------------------------------------------------

[FRAGMENTO 2]
Fuente Real: informe CENIA CEPAL 2025.pdf | Pág: 174
Contenido:
ILIA 2025 174 y un tercero que mide la proporción de energías renovables no convencionales (ERNC) dentro de la matriz energética. Con estas modificaciones, el ILIA 2025 b

### Búsqueda por Umbral de Similitud (Similarity Score Threshold)

**Para evitar que el sistema procese basura, podemos configurar un retriever que solo devuelva fragmentos si superan un umbral mínimo de similitud (56 a 62)**

In [31]:
import os
from langchain_community.vectorstores import Chroma

# --- 1. CONFIGURACIÓN DEL TEST ---
# IMPORTANTE: Reemplazá con el nombre de tu corpus (sin .json)
CORPUS_A_TESTEAR = "Informe_Sustentabilidad_2025" 

query_coherente = "Desarrollo sostenible y metas para el 2030"
query_fuera_contexto = "¿Cuál es la alineación de la selección argentina?"

# Umbral mínimo de similitud (0.62 = 62% de coincidencia semántica)
UMBRAL = 0.62 

ruta_vector_db = os.path.join("chroma_db", CORPUS_A_TESTEAR)

# Verificamos que la base de datos exista
if not os.path.exists(ruta_vector_db):
    print(f"Error: No se encontró la base de datos en {ruta_vector_db}")
else:
    # --- 2. CONEXIÓN A CHROMA ---
    print(f"[INFO] Conectando a la base de datos de '{CORPUS_A_TESTEAR}'...")
    
    vector_store_test = Chroma(
        persist_directory=ruta_vector_db, 
        embedding_function=embeddings_model
    )
    
    # --- 3. CONFIGURACIÓN DEL RETRIEVER CON UMBRAL ---
    retriever_umbral = vector_store_test.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "k": 3,
            "score_threshold": UMBRAL 
        }
    )

    # --- 4. TEST 1: PREGUNTA RELEVANTE ---
    print("\n" + "="*80)
    print("TEST 1: Pregunta coherente con el corpus")
    print(f"Query: '{query_coherente}'")
    
    try:
        docs_validos = retriever_umbral.invoke(query_coherente)
        print(f"📄 Fragmentos recuperados que superan el umbral ({UMBRAL}): {len(docs_validos)}")
        
        if len(docs_validos) > 0:
            print("\n[Muestra del mejor fragmento]")
            print(f"Fuente: {docs_validos[0].metadata.get('fuente')} | Pág: {docs_validos[0].metadata.get('pagina')}")
            print(f"Contenido: {docs_validos[0].page_content[:200]}...")
    except Exception as e:
        # A veces LangChain lanza una advertencia si no encuentra NADA que supere el umbral
        print(f"Resultado: No se superó el umbral. (Error/Warning: {e})")

    # --- 5. TEST 2: PREGUNTA IRRELEVANTE ---
    print("\n" + "="*80)
    print("TEST 2: Pregunta fuera de contexto")
    print(f"Query: '{query_fuera_contexto}'")
    
    try:
        docs_fuera = retriever_umbral.invoke(query_fuera_contexto)
        print(f"Fragmentos recuperados que superan el umbral ({UMBRAL}): {len(docs_fuera)}")
        if len(docs_fuera) == 0:
            print("¡Éxito! El sistema filtró correctamente la pregunta irrelevante.")
    except Exception as e:
        print("¡Éxito! El sistema filtró correctamente la pregunta irrelevante bloqueando la búsqueda.")
    
    print("="*80)

No relevant docs were retrieved using the relevance score threshold 0.62


[INFO] Conectando a la base de datos de 'Informe_Sustentabilidad_2025'...

TEST 1: Pregunta coherente con el corpus
Query: 'Desarrollo sostenible y metas para el 2030'
📄 Fragmentos recuperados que superan el umbral (0.62): 3

[Muestra del mejor fragmento]
Fuente: informe CENIA CEPAL 2025.pdf | Pág: 166
Contenido: el indicador de Ética y Sustentabilidad se compone de información del Índice Global de IA Responsable (GIRAI), del Índice de Preparación de Redes (NRI) y otros elementos de sustentabilidad relacionado...

TEST 2: Pregunta fuera de contexto
Query: '¿Cuál es la alineación de la selección argentina?'
Fragmentos recuperados que superan el umbral (0.62): 0
¡Éxito! El sistema filtró correctamente la pregunta irrelevante.


# 6. Interfaz Gráfica (Gradio)
Levantamos la UI construida en pestañas para interactuar con todas las funciones del backend.

In [27]:
# --- CSS PERSONALIZADO ---
# Acá definimos los colores. Usé un fondo celeste muy clarito con un borde azul.
# Podés cambiar los códigos de color (Hex) si preferís otro estilo.
css_estilos = """
#caja-chat textarea {
    background-color: #F0F8FF !important; /* Fondo AliceBlue (celeste muy claro) */
    border: 2px solid #4A90E2 !important; /* Borde azul fuerte para que resalte */
    font-size: 16px !important; /* Letra un poco más grande para leer mejor */
    padding: 15px !important; /* Más espacio interno para que no quede apretado */
    border-radius: 8px !important; /* Bordes redondeados */
}
"""

# Pasamos el CSS al bloque principal de Gradio
with gr.Blocks(theme=gr.themes.Soft(), css=css_estilos) as demo:
    gr.Markdown("# 🤖 Plataforma RAG Local Avanzada (Multicorpus)")
    
    lista_inicial = listar_corpus_disponibles()
    
    with gr.Tabs():
        # PESTAÑA 1: GESTIÓN DE DOCUMENTOS
        with gr.Tab("📁 Gestión de Corpus"):
            with gr.Row():
                with gr.Column():
                    archivos_pdf = gr.File(label="Subir PDFs", file_count="multiple", file_types=[".pdf"])
                    accion_radio = gr.Radio(["Crear nuevo corpus", "Agregar a corpus existente"], label="Acción", value="Crear nuevo corpus")
                with gr.Column():
                    nombre_nuevo = gr.Textbox(label="Nombre nuevo corpus", visible=True)
                    dropdown_existente_gestion = gr.Dropdown(choices=lista_inicial, label="Seleccionar corpus", visible=False)
                    btn_procesar = gr.Button("⚙️ Procesar y Guardar JSON", variant="primary")
            mensaje_gestion = gr.Textbox(label="Estado de carga", interactive=False)
            
            gr.Markdown("---")
            with gr.Row():
                dropdown_eliminar = gr.Dropdown(choices=lista_inicial, label="Seleccionar a eliminar", scale=3)
                btn_eliminar = gr.Button("Eliminar Definitivamente", variant="stop", scale=1)
            mensaje_eliminar = gr.Textbox(label="Estado", interactive=False)
            
            def actualizar_visibilidad(accion):
                if accion == "Crear nuevo corpus": return gr.update(visible=True), gr.update(visible=False)
                else: return gr.update(visible=False), gr.update(visible=True)
            accion_radio.change(fn=actualizar_visibilidad, inputs=accion_radio, outputs=[nombre_nuevo, dropdown_existente_gestion])
            
        # PESTAÑA 2: CONFIGURACIÓN RAG
        with gr.Tab("⚙️ Configuración del RAG"):
            with gr.Row():
                dropdown_cargar_rag = gr.Dropdown(choices=lista_inicial, label="Corpus disponible")
                btn_cargar_rag = gr.Button("🚀 Cargar al Sistema RAG", variant="primary")
            mensaje_carga = gr.Textbox(label="Estado del Sistema RAG", interactive=False)
            btn_cargar_rag.click(fn=cargar_corpus_en_rag, inputs=dropdown_cargar_rag, outputs=mensaje_carga)

        # PESTAÑA 3: CHAT (Modificada)
        with gr.Tab("💬 Chat"):
            gr.Markdown("### 💬 Consulta Inteligente de Documentos")
            chatbot = gr.ChatInterface(
                fn=responder_rag_interactivo,
                textbox=gr.Textbox(
                    placeholder="✍️ Escribí tu consulta aquí y presioná Enter...", 
                    container=True, 
                    scale=7,
                    elem_id="caja-chat" # Este ID conecta la caja con nuestro CSS personalizado
                ),
            )

        # Eventos
        btn_procesar.click(
            fn=procesar_pdf, 
            inputs=[archivos_pdf, nombre_nuevo, dropdown_existente_gestion, accion_radio], 
            outputs=[mensaje_gestion, dropdown_existente_gestion, dropdown_eliminar, dropdown_cargar_rag]
        )
        btn_eliminar.click(
            fn=eliminar_corpus,
            inputs=dropdown_eliminar,
            outputs=[mensaje_eliminar, dropdown_existente_gestion, dropdown_eliminar, dropdown_cargar_rag]
        )

# Ejecutamos la app
demo.launch(share=True, debug=True)

C:\Users\admin\AppData\Local\Temp\ipykernel_16228\4216904895.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=css_estilos) as demo:


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


d:\job\TECNICATURA\CD e IA 2 2026\Procesamiento del habla\barrionuevo-jose-pln-1c-2026-tp-integrador\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\job\TECNICATURA\CD e IA 2 2026\Procesamiento del habla\barrionuevo-jose-pln-1c-2026-tp-integrador\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\job\TECNICATURA\CD e IA 2 2026\Procesamiento del habla\barrionuevo-jose-pln-1c-2026-tp-integrador\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\job\TECNICATU

Keyboard interruption in main thread... closing server.
